# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (use attributes, not dictionary subscripting as per mlcroissant usage)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All browsing is performed via `@id` fields for reproducibility and reference integrity.

Let's enumerate the available record sets and their fields in the dataset.

In [ ]:
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet ID (@id): {rs.id}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print(f"  Number of fields: {len(rs.fields)}")
    for f in rs.fields:
        print(f"    Field @id: {f.id}, name: {getattr(f, 'name', None)}, dataType: {getattr(f, 'data_type', None)}")
    print("")
if not record_sets:
    print("No record sets found in this dataset's Croissant schema.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame. All references are by their `@id` fields.

For this dataset, we scan for record sets (tables) and extract their records. If no record sets are found, this means the dataset does not supply data in individual tabular form attached as Croissant record sets, and you may need to fetch data another way.

In [ ]:
dataframes = {}
if record_sets:
    record_set_ids = [rs.id for rs in record_sets]
    for rs in record_sets:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"DataFrame for record set '{rs.id}' with {df.shape[0]} rows and {df.shape[1]} columns loaded.")

    # Pick the first record set for demonstration
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in '{first_rs_id}': {dataframes[first_rs_id].columns.tolist()}")
    # Show first few records
    display(dataframes[first_rs_id].head())
else:
    print("No tabular record sets found. Cannot extract data into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping. This section refers to fields by their full `@id`, as displayed in the overview above.

In [ ]:
# For EDA, pick the first available DataFrame with numeric fields
import numpy as np
if dataframes:
    df = dataframes[first_rs_id]

    # Attempt to automatically pick a numeric field (float or int) using Field objects from the record set
    numeric_field_id = None
    group_field_id = None
    for f in record_sets[0].fields:
        dtype = getattr(f, "data_type", None)
        if dtype in ["https://schema.org/Float", "https://schema.org/Integer"] and f.id in df.columns:
            numeric_field_id = f.id
            break
    for f in record_sets[0].fields:
        if getattr(f, "data_type", None) == "https://schema.org/Text" and f.id in df.columns:
            group_field_id = f.id
            break

    if numeric_field_id is not None:
        # Convert column to numeric (handling NaN, etc.)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors="coerce")
        threshold = df[numeric_field_id].mean()  # Use mean as threshold exemplary
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalize the selected field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by the group field (if available)
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable text group field found for grouping.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No tabular data to perform EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and (optionally) a grouped summary, using `matplotlib`, if numeric fields are present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        # Plot mean by group (top 10 groups only)
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped.index, y=grouped.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (top 10)")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset, using `mlcroissant` to enumerate its record sets and fields, extract records by `@id`, and perform basic exploratory data analysis and visualization. For in-depth study, you are encouraged to refer to field `@id`s found programmatically above, and adapt the filtering and grouping as suits your research questions.